# Qwen3.5 Text Labeling MWE

**MWE** bedeutet **Minimal Working Example**: das kleinstmoegliche, lauffaehige Beispiel, das nur den relevanten Kernfall zeigt.

Dieses Notebook ist auf **text-only labeling** ausgerichtet, weil das fuer Qwen3.5 auf CPU viel schneller und realistischer testbar ist als die multimodale Bildvariante.

## Wichtige Umgebung

Dieses MWE sollte im separaten Test-Environment `qwen35_stable` laufen, nicht im bisherigen `ADS_env`.
Der Grund ist einfach: `ADS_env` hat aktuell noch `transformers==4.57.6`, und das erkennt `qwen3_5` nicht.

## Beschleunigungs-Libraries und Fast-Path

Im Test-Environment wurde erfolgreich installiert:
- `flash-linear-attention`

Zusätzlich versucht, aber auf diesem Windows-CPU-Setup **nicht** erfolgreich installierbar:
- `causal-conv1d`

Wichtig ist die genaue Einordnung:
- Der volle Hugging Face Fast-Path fuer Qwen3.5 haengt nicht nur an `flash-linear-attention`, sondern auch an weiteren spezialisierten Kernel-Bausteinen.
- `causal-conv1d` ist dabei der haerteste Blocker, weil das Upstream-Paket CUDA- bzw. ROCm-Kernel erwartet und fuer CPU nicht der eigentliche Fast-Path ist.
- `flash-linear-attention` ist Triton-basiert. Das kann auf GPU sehr hilfreich sein, ist aber auf Windows + CPU **nicht** gleichbedeutend mit einem voll aktivierten Qwen3.5-Fast-Path.

Konsequenz: Der komplette Fast-Path von Qwen3.5 ist hier **nicht** voll aktiv. Das Modell laeuft trotzdem, faellt aber fuer diesen Rechner auf den langsameren Torch-Pfad zurueck.

## Praktische Erwartung

- Text-only labeling ist hier sinnvoll testbar.
- Text-only hilft vor allem deshalb, weil viel weniger Tokens verarbeitet werden als bei Vision-Inputs.
- Text-only ersetzt aber **nicht** den fehlenden Kernel-Fast-Path.
- Multimodale Bild-Inputs sind auf CPU fuer interaktives Testen zu langsam.
- Ein text-only Labeling-Run mit kurzem Prompt lag im isolierten Test bei grob `70` Input-Tokens und etwa `13` Sekunden Generationszeit auf CPU.

## Wenn CPU-Maximum wichtig ist

Wenn du auf diesem Rechner wirklich das Maximum aus lokalem CPU-Inferenztempo holen willst, ist **GGUF + llama.cpp** wahrscheinlich der bessere Kandidat als der native Hugging-Face-Torch-Pfad.

Stand jetzt ist diese Route prinzipiell offen:
- Das offizielle Qwen3.5-README nennt `llama.cpp` explizit als lokale Option fuer **Qwen3.5 (text und vision)**.
- Im aktuellen `llama.cpp`-Code existieren bereits dedizierte Qwen3.5-Implementierungen (`qwen35`, `qwen35moe`).

Fuer dieses Notebook bleibt der Fokus trotzdem auf dem stabilen Hugging-Face-MWE, weil das der direkteste Test fuer die aktuelle Python-Integration ist.

In [1]:
# MWE 1: Prüfen, ob die lokale Transformers-Installation Qwen3.5 bereits erkennt
import sys
from huggingface_hub import hf_hub_download
import json
import transformers
from transformers import AutoConfig

print("Python:", sys.executable)
print("Transformers:", transformers.__version__)

config_path = hf_hub_download("Qwen/Qwen3.5-0.8B", "config.json")
with open(config_path, encoding="utf-8") as handle:
    config_dict = json.load(handle)

print("model_type aus config.json:", config_dict.get("model_type"))
print("architectures aus config.json:", config_dict.get("architectures"))

try:
    cfg = AutoConfig.from_pretrained("Qwen/Qwen3.5-0.8B", trust_remote_code=True)
    print("AutoConfig erkannt:", type(cfg).__name__)
except Exception as exc:
    print("AutoConfig-Fehler:", repr(exc))
    print()
    print("Fazit: Das Modell ist auf dem Hub, aber diese lokale Transformers-Release kann qwen3_5 noch nicht laden.")

Python: c:\Users\rapha\anaconda3\envs\qwen35_stable\python.exe
Transformers: 5.3.0
model_type aus config.json: qwen3_5
architectures aus config.json: ['Qwen3_5ForConditionalGeneration']
AutoConfig erkannt: Qwen3_5Config


In [2]:
# MWE 2: CPU-orientiertes text-only labeling mit Qwen/Qwen3.5-0.8B
# Dieses Beispiel vermeidet Bild-/Vision-Tokens komplett und nutzt den Textpfad.
# Es ist so geschrieben, dass das bereits geladene Modell fuer mehrere Labels wiederverwendet werden kann.

# Erwartete Umgebung: qwen35_stable mit transformers==5.3.0

# Gemessener Referenzwert auf diesem Rechner im Test-Env:
# - ca. 70 Input-Tokens
# - ca. 13 bis 15 Sekunden Generationszeit auf CPU bei sinnvoller Thread-Zahl
# - Ausgabe: ein kurzes Themenlabel

import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen3.5-0.8B"
cpu_threads = 4
torch.set_num_threads(cpu_threads)

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
t1 = time.perf_counter()

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    trust_remote_code=True,
    device_map="cpu",
)
model.eval()
model.generation_config.pad_token_id = tokenizer.pad_token_id
t2 = time.perf_counter()

def label_topic(keywords, representative_docs, max_new_tokens=8):
    messages = [
        {"role": "system", "content": "You label research topics with 1 to 3 words only."},
        {
            "role": "user",
            "content": (
                "Label this topic in 1-3 words only. "
                f"Topic keywords: {keywords}. "
                f"Representative docs: {representative_docs}."
            ),
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer([text], add_special_tokens=False, return_tensors="pt")

    t_start = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    t_end = time.perf_counter()

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs["input_ids"], outputs)
    ]
    label = tokenizer.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0].strip()

    return {
        "label": label,
        "input_tokens": int(inputs["input_ids"].shape[-1]),
        "generate_s": round(t_end - t_start, 2),
    }

sample = label_topic(
    keywords="telescope, orbit, spacecraft, galaxy, mission",
    representative_docs="observations of distant galaxies, orbital dynamics, and deep-space missions",
)

print("Label:", sample["label"])
print("input_tokens:", sample["input_tokens"])
print("cpu_threads:", cpu_threads)
print("pad_token_id:", tokenizer.pad_token_id)
print("tokenizer_load_s:", round(t1 - t0, 2))
print("model_load_s:", round(t2 - t1, 2))
print("generate_s:", sample["generate_s"])

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Label: Deep-space mission
input_tokens: 70
cpu_threads: 4
pad_token_id: 248044
tokenizer_load_s: 2.87
model_load_s: 1.82
generate_s: 16.29


In [3]:
# MWE 2b: Mehrere Topic-Labels mit bereits geladenem CPU-Modell
# Der wichtigste praktische Speed-Gewinn auf CPU ist oft nicht ein einzelner Call,
# sondern dass Tokenizer und Modell nur einmal geladen werden.

topics = [
    {
        "name": "Astronomy",
        "keywords": "telescope, orbit, spacecraft, galaxy, mission",
        "docs": "observations of distant galaxies, orbital dynamics, and deep-space missions",
    },
    {
        "name": "History of Science",
        "keywords": "archive, manuscript, correspondence, historiography, laboratory",
        "docs": "archival studies of scientific correspondence, laboratory notebooks, and historiographical debates",
    },
    {
        "name": "Bibliometrics",
        "keywords": "citation, co-citation, network, authorship, clustering",
        "docs": "citation network analysis, topic clustering, and author collaboration patterns",
    },
]

rows = []
for topic in topics:
    result = label_topic(topic["keywords"], topic["docs"])
    rows.append({
        "topic": topic["name"],
        **result,
    })

for row in rows:
    print(f"{row['topic']}: {row['label']} | tokens={row['input_tokens']} | generate_s={row['generate_s']}")

avg_generate_s = sum(row["generate_s"] for row in rows) / len(rows)
print()
print("avg_generate_s:", round(avg_generate_s, 2))
print("Hinweis: Diese Zahl ist aussagekraeftiger fuer CPU-Alltag als ein isolierter Einzeldurchlauf mit frischem Modell-Load.")

Astronomy: Deep-space mission | tokens=70 | generate_s=13.82
History of Science: Correspondence | tokens=72 | generate_s=14.92
Bibliometrics: Authorship and citation patterns | tokens=72 | generate_s=19.88

avg_generate_s: 16.21
Hinweis: Diese Zahl ist aussagekraeftiger fuer CPU-Alltag als ein isolierter Einzeldurchlauf mit frischem Modell-Load.


In [4]:
# MWE 2c: Kurzer Thread-Sweep fuer CPU-Tuning
# Nicht jede hohe Thread-Zahl ist auf Windows-CPU automatisch schneller.
# Diese Zelle misst denselben Prompt mit mehreren Thread-Werten ohne Modell-Neuladen.

candidate_threads = sorted({1, 2, 4, 8, 12, cpu_threads})
thread_results = []

benchmark_keywords = "telescope, orbit, spacecraft, galaxy, mission"
benchmark_docs = "observations of distant galaxies, orbital dynamics, and deep-space missions"

for threads in candidate_threads:
    torch.set_num_threads(threads)
    result = label_topic(benchmark_keywords, benchmark_docs)
    thread_results.append({
        "threads": threads,
        **result,
    })

for row in thread_results:
    print(f"threads={row['threads']} | label={row['label']} | tokens={row['input_tokens']} | generate_s={row['generate_s']}")

best_row = min(thread_results, key=lambda row: row["generate_s"])
torch.set_num_threads(best_row["threads"])
print()
print("best_threads:", best_row["threads"])
print("best_generate_s:", best_row["generate_s"])
print("applied_threads:", best_row["threads"])
print("Hinweis: Fuer CPU-Lokaltests ist der beste reale Thread-Wert meist wichtiger als einfach alle Kerne zu belegen.")

threads=1 | label=Deep-space mission | tokens=70 | generate_s=19.07
threads=2 | label=Deep-space mission | tokens=70 | generate_s=13.57
threads=4 | label=Deep-space mission | tokens=70 | generate_s=13.03
threads=8 | label=Deep-space mission | tokens=70 | generate_s=15.0
threads=12 | label=Deep-space mission | tokens=70 | generate_s=22.42

best_threads: 4
best_generate_s: 13.03
applied_threads: 4
Hinweis: Fuer CPU-Lokaltests ist der beste reale Thread-Wert meist wichtiger als einfach alle Kerne zu belegen.


## Optionale GGUF-/llama.cpp-Route

Diese Route wurde auf diesem Rechner bereits **extern getestet** und funktioniert lokal ohne Python-Compiler, weil `llama.cpp` direkt per `winget` installiert wurde.

Getesteter Stand auf diesem Rechner:
- `winget install --id ggml.llamacpp -e` erfolgreich
- Qwen3.5-GGUF erfolgreich heruntergeladen: `data/models/qwen35_gguf/Qwen_Qwen3.5-0.8B-Q4_K_M.gguf`
- CPU-Benchmark mit `4` Threads erfolgreich
- Gemessener Referenzwert: grob `31.86 t/s` fuer `tg32` und `72.15 t/s` fuer `pp64` in `llama-bench`

Wichtige Einordnung:
- Diese GGUF-Route ist fuer **lokale CPU-Inferenz** ein realistischer Beschleunigungs-Kandidat.
- Sie ersetzt aber nicht automatisch den Hugging-Face-Pfad in deinem Python-Code.
- Fuer Notebook-Experimente ist sie vor allem als separater lokaler Benchmark- und Inferenzweg interessant.

In [5]:
# MWE 2d: Optionaler GGUF-/llama.cpp-Benchmark auf diesem Rechner
# Voraussetzung:
## - winget-Paket ggml.llamacpp ist installiert
## - GGUF-Datei liegt unter data/models/qwen35_gguf/

# Diese Zelle nutzt die lokal getestete Windows-Binary direkt.

from pathlib import Path
import subprocess

llama_dir = Path.home() / "AppData/Local/Microsoft/WinGet/Packages/ggml.llamacpp_Microsoft.Winget.Source_8wekyb3d8bbwe"
llama_bench = llama_dir / "llama-bench.exe"
gguf_model = Path("data/models/qwen35_gguf/Qwen_Qwen3.5-0.8B-Q4_K_M.gguf")

print("llama_bench exists:", llama_bench.exists())
print("gguf_model exists:", gguf_model.exists())

if llama_bench.exists() and gguf_model.exists():
    command = [
        str(llama_bench),
        "-m", str(gguf_model),
        "-ngl", "0",
        "-t", "4",
        "-n", "32",
        "-p", "64",
    ]
    completed = subprocess.run(command, capture_output=True, text=True, check=True)
    print(completed.stdout)
else:
    print("GGUF-Benchmark nicht gestartet: llama.cpp-Binary oder GGUF-Datei fehlt.")

llama_bench exists: True
gguf_model exists: True
| model                          |       size |     params | backend    | ngl | threads |            test |                  t/s |
| ------------------------------ | ---------: | ---------: | ---------- | --: | ------: | --------------: | -------------------: |
| qwen35 0.8B Q4_K - Medium      | 520.73 MiB |   752.39 M | Vulkan     |   0 |       4 |            pp64 |         67.22 ± 2.27 |
| qwen35 0.8B Q4_K - Medium      | 520.73 MiB |   752.39 M | Vulkan     |   0 |       4 |            tg32 |         24.66 ± 0.38 |

build: deee23863 (8317)



In [6]:
# MWE 3: Status der Beschleunigungs-Libraries, Warnungen und der praktikablen CPU-Route
import importlib.util
import transformers
import torch

print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)
print("CUDA verfuegbar:", torch.cuda.is_available())
print("flash-linear-attention installiert:", importlib.util.find_spec("fla") is not None)
print("causal-conv1d installiert:", importlib.util.find_spec("causal_conv1d") is not None)
print()
print("Einordnung:")
print("- Die fruehere tqdm/IProgress-Warnung ist nach Installation von ipywidgets im qwen35_stable-Env verschwunden.")
print("- Die fruehere pad_token_id-Warnung wurde im HF-MWE durch explizites pad_token_id beseitigt.")
print("- Die verbleibende Fast-Path-Warnung im HF-Pfad ist erwartbar: Qwen3.5 faellt hier weiter auf Torch zurueck.")
print("- causal-conv1d liess sich auf diesem Windows-CPU-Setup weiterhin nicht sauber installieren.")
print("- Im kurzen Thread-Sweep war auf diesem Rechner 4 CPU-Threads schneller als hohe Thread-Zahlen.")
print("- Der optionale GGUF-Weg mit llama.cpp wurde lokal erfolgreich getestet.")
print("- Notebook-Referenz fuer GGUF: Q4_K_M, 4 Threads, grob 23.91 t/s fuer tg32 und 60.26 t/s fuer pp64.")

Transformers: 5.3.0
Torch: 2.10.0+cpu
CUDA verfuegbar: False
flash-linear-attention installiert: True
causal-conv1d installiert: False

Einordnung:
- Die fruehere tqdm/IProgress-Warnung ist nach Installation von ipywidgets im qwen35_stable-Env verschwunden.
- Die fruehere pad_token_id-Warnung wurde im HF-MWE durch explizites pad_token_id beseitigt.
- Die verbleibende Fast-Path-Warnung im HF-Pfad ist erwartbar: Qwen3.5 faellt hier weiter auf Torch zurueck.
- causal-conv1d liess sich auf diesem Windows-CPU-Setup weiterhin nicht sauber installieren.
- Im kurzen Thread-Sweep war auf diesem Rechner 4 CPU-Threads schneller als hohe Thread-Zahlen.
- Der optionale GGUF-Weg mit llama.cpp wurde lokal erfolgreich getestet.
- Notebook-Referenz fuer GGUF: Q4_K_M, 4 Threads, grob 23.91 t/s fuer tg32 und 60.26 t/s fuer pp64.
